# MoSAIC-LiITA Demo

This notebook demonstrates both translation modes:
- **Deterministic**: Rule-based planner using keyword matching
- **Agentic**: LLM-powered query decomposition

**MoSAIC** — **Mo**dular **S**parql **A**ssembler for **I**nterlinked **C**orpora

## Setup

In [ ]:
import json
from pathlib import Path

import requests

from mosaic_liita import (
    Planner,
    Assembler,
    QueryAgent,
    make_registry,
)
from shared.llm import create_llm_client

In [ ]:
# Configuration
DATA_DIR = Path("data")
ONTOLOGY_PATH = DATA_DIR / "ontology_filtered.json"
LIITA_ENDPOINT = "https://liita.it/sparql"

In [ ]:
# Load catalog and initialize components
with open(ONTOLOGY_PATH, "r", encoding="utf-8") as f:
    catalog = json.load(f)["documents"]

registry = make_registry()
planner = Planner(registry, catalog)
assembler = Assembler()

print(f"Loaded {len(catalog)} catalog entries")
print(f"Registry contains {len(registry.blocks)} blocks")

In [ ]:
registry.blocks['TRANSLATE_TO_SICILIANO']

In [ ]:
def execute_sparql(sparql: str, limit: int = 10) -> dict:
    """Execute a SPARQL query against the LiITA endpoint."""
    if "LIMIT" not in sparql.upper():
        sparql = sparql.rstrip().rstrip(";") + f"\nLIMIT {limit}"
    
    response = requests.post(
        LIITA_ENDPOINT,
        data={"query": sparql},
        headers={
            "Accept": "application/sparql-results+json",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def display_results(data: dict):
    """Display SPARQL results in a readable format."""
    variables = data.get("head", {}).get("vars", [])
    bindings = data.get("results", {}).get("bindings", [])
    
    print(f"Results: {len(bindings)} rows")
    print(f"Variables: {', '.join(variables)}\n")
    
    for i, row in enumerate(bindings, 1):
        print(f"--- Row {i} ---")
        for var in variables:
            if var in row:
                val = row[var].get("value", "")
                print(f"  {var}: {val}")
        print()

---

## Deterministic Mode

The deterministic planner uses rule-based pattern matching to select blocks. No LLM required.

In [ ]:
def translate_deterministic(question: str) -> str:
    """Translate a natural language question to SPARQL using the deterministic planner."""
    spec = planner.plan(question)
    plan = spec.compile(registry)
    sparql = assembler.assemble(plan)
    
    # Show plan details
    print("=" * 60)
    print(f"Query: {question}")
    print("=" * 60)
    print(f"\nBlocks used ({len(plan.blocks)}):")
    for i, bi in enumerate(plan.blocks, 1):
        print(f"  {i}. {bi.block.id}")
        if bi.slots:
            for k, v in bi.slots.items():
                if v:
                    print(f"     - {k}: {v}")
    
    print(f"\nSelect variables: {plan.select_vars}")
    if plan.aggregates:
        print(f"Aggregates: {plan.aggregates}")
    
    print("\n" + "-" * 60)
    print("Generated SPARQL:")
    print("-" * 60)
    print(sparql)
    
    return sparql

### Example 1: Find synonyms of a word

In [ ]:
sparql = translate_deterministic("Find synonyms of 'origine'")

In [ ]:
# Execute the query
results = execute_sparql(sparql)
display_results(results)

### Example 2: Find definitions by pattern

In [ ]:
sparql = translate_deterministic("Find definitions of words starting with 'ante'")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 3: Translation to dialect

In [ ]:
sparql = translate_deterministic("Find Sicilian lemmas ending with 'u' and translate them into Italian")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 4: Emotions and semantic relations

In [ ]:
sparql = translate_deterministic(
    "Find Italian words that express positive emotions ('gioia') and are hyponyms of 'sentimento'"
)

In [ ]:
results = execute_sparql(sparql)
display_results(results)

---

## Agentic Mode

The agentic translator uses an LLM to decompose queries into tool operations.

**Requirements:** Configure your LLM provider and API key below.

In [ ]:
# LLM Configuration
# Choose your provider: "mistral", "anthropic", "openai", "gemini", "ollama"

LLM_PROVIDER = "ollama"  # Change this to your provider
LLM_API_KEY = ""         # Not needed for Ollama
LLM_MODEL = "gemini-3-flash-preview:cloud"   # Or leave empty for default

# Default models per provider:
# - mistral: "mistral-large-latest"
# - anthropic: "claude-sonnet-4-20250514"
# - openai: "gpt-4o"
# - gemini: "gemini-1.5-pro"
# - ollama: "llama3.1"

In [ ]:
# Initialize LLM client and agent
llm_client = create_llm_client(
    provider=LLM_PROVIDER,
    api_key=LLM_API_KEY if LLM_API_KEY else None,
    model=LLM_MODEL,
)

agent = QueryAgent(registry, catalog, llm_client)
print(f"Agent initialized with {LLM_PROVIDER} ({LLM_MODEL or 'default model'})")

In [ ]:
def translate_agentic(question: str) -> str:
    """Translate a natural language question to SPARQL using the agentic planner."""
    sparql, agent_plan, query_spec = agent.translate(question)
    
    print("=" * 60)
    print(f"Query: {question}")
    print("=" * 60)
    
    print(f"\nAgent Reasoning:")
    print(f"  {agent_plan.reasoning}")
    
    print(f"\nTool Calls ({len(agent_plan.steps)}):")
    for i, step in enumerate(agent_plan.steps, 1):
        print(f"  {i}. {step.tool}")
        if step.params:
            for k, v in step.params.items():
                print(f"     - {k}: {v}")
    
    print(f"\nOutput variables: {agent_plan.output_vars}")
    
    if agent_plan.aggregation:
        print(f"Aggregation: {agent_plan.aggregation}")
    
    print("\n" + "-" * 60)
    print("Generated SPARQL:")
    print("-" * 60)
    print(sparql)
    
    return sparql

### Example 1: Complex query with filtering

In [ ]:
sparql = translate_agentic("Find synonyms of 'origine' whose Sicilian translation starts with 'm'")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 2: Counting results

In [ ]:
sparql = translate_agentic("How many lemmas are present in the Sicilian lexicon?")

In [ ]:
results = execute_sparql(sparql)
display_results(results)

### Example 3: Multi-step translation

In [ ]:
sparql = translate_agentic(
    "Find CompL-IT definitions starting with 'uccello' and show Parmigiano and Sicilian translations"
)

In [ ]:
results = execute_sparql(sparql)
display_results(results)

---

## Comparison

Try the same query with both modes to see the differences.

In [ ]:
test_query = "Find synonyms of 'origine' with their Sicilian translations"

print("DETERMINISTIC MODE")
print("=" * 60)
sparql_det = translate_deterministic(test_query)

In [ ]:
print("\nAGENTIC MODE")
print("=" * 60)
sparql_agent = translate_agentic(test_query)

In [ ]:
# Compare results
print("Deterministic results:")
display_results(execute_sparql(sparql_det))

print("\nAgentic results:")
display_results(execute_sparql(sparql_agent))